# Open Action Designator

Demonstrates `OpenActionDescription` — opening a drawer or door by its handle body.

The robot must be pre-navigated to a valid base pose in front of the handle before opening.

**Two approaches:**
- **Direct PyCRAM** — navigate + open with explicit handle body
- **LLM Pipeline** — natural language → `run_with_cache` → CRAM `(type Opening)` → `SimulationBridge`

**Prerequisites:** `cd cognitive_robot_abstract_machine && uv sync --active`

## 1 · Imports

In [ ]:
import logging
logging.basicConfig(level=logging.WARNING)

from semantic_digital_twin.world import World
from semantic_digital_twin.robots.pr2 import PR2
from semantic_digital_twin.datastructures.definitions import TorsoState

from pycram.datastructures.dataclasses import Context
from pycram.datastructures.enums import Arms
from pycram.datastructures.pose import PoseStamped
from pycram.language import SequentialPlan
from pycram.motion_executor import simulated_robot
from pycram.testing import setup_world
from pycram.robot_plans import (
    MoveTorsoActionDescription,
    ParkArmsActionDescription,
    NavigateActionDescription,
    OpenActionDescription,
)

from llmr.adapters import SimulationBridge
from llmr.workflows.graphs.ad_graph import run_with_cache

print('All imports OK')

## 2 · World & Context Setup

In [ ]:
world = setup_world()
context = Context(world, robot)

## 3 · SimulationBridge Setup

In [ ]:
bridge = SimulationBridge(world, robot)

## 4 · Visualize in RViz2 (optional)

Skip this section if ROS2 is not available.

In [ ]:
from semantic_digital_twin.adapters.ros.tf_publisher import TFPublisher
from semantic_digital_twin.adapters.ros.visualization.viz_marker import VizMarkerPublisher
import threading
import rclpy

rclpy.init()
_ros_node = rclpy.create_node('semantic_digital_twin')
_ros_thread = threading.Thread(target=rclpy.spin, args=(_ros_node,), daemon=True)
_ros_thread.start()

In [ ]:
_tf_publisher  = TFPublisher(_world=world, node=_ros_node)
_viz_publisher = VizMarkerPublisher(_world=world, node=_ros_node)

---
## 5 · Target Handle & Navigation Pose

We target `handle_cab10_t` (top drawer of cabinet 10).  
The navigation pose is pre-computed to place the robot in front of the drawer.

In [ ]:
handle_body = world.get_body_by_name('handle_cab10_t')
open_nav    = PoseStamped.from_list(
    [1.7474915981292725, 2.6873629093170166, 0.0],
    [-0.0, 0.0, 0.5253598267689507, -0.850880163370435],
    world.root,
)
print(f'Handle body: {handle_body}')
print(f'Nav pose   : {open_nav}')

---
## 6 · Open — Direct PyCRAM

In [ ]:
with simulated_robot:
    SequentialPlan(
        context,
        MoveTorsoActionDescription([TorsoState.HIGH]),
        ParkArmsActionDescription([Arms.BOTH]),
        NavigateActionDescription([open_nav]),
        OpenActionDescription(handle_body, [Arms.RIGHT]),
    ).perform()

print('Open (direct) done')

---
## 7 · Open — LLM Pipeline

Natural language → `run_with_cache` → CRAM `(type Opening)` → `execute_batch`

We pre-navigate the robot before handing over to the bridge.

In [ ]:
instruction = 'Open the handle_cab10_t drawer.'
result      = run_with_cache(instruction, use_cache=False)
cram_plans  = result.get('cram_plan_response', [])

print(f'LLM generated {len(cram_plans)} plan(s):')
for i, p in enumerate(cram_plans, 1):
    print(f'  Plan {i}: {p}')

with simulated_robot:
    results = bridge.execute_batch(cram_plans, arm=Arms.RIGHT)

print('\nOpen (LLM pipeline) done  results:', results)

---
## Shutdown ROS2 Node

In [ ]:
try:
    _ros_node.destroy_node()
    rclpy.shutdown()
    print('ROS2 node stopped')
except Exception:
    print('(ROS2 node was not running)')